# EB Causal AUX SPINN Forward Problem (Demo)

Euler-Bernoulli beam with auxiliary variable $\nu(x,t) = u_{xx}$.
Splits the 4th-order operator into two 2nd-order relationships:

$$
\nu - u_{xx} = 0 \quad\text{(consistency, $\lambda_v = 10$)}
$$
$$
u_{tt} + \nu_{xx} + (\pi^2 - 1)\,u = 0 \quad\text{(PDE)}
$$


In [ ]:
import jax
jax.config.update('jax_default_matmul_precision', 'float32')

import jax.numpy as jnp
import numpy as np
import optax
from jax import jvp, value_and_grad
from flax import linen as nn
from typing import Sequence
from functools import partial
from tqdm.auto import trange
import matplotlib.pyplot as plt


## Hyperparameters

In [ ]:
SEED       = 111
NC         = 128
NC_TEST    = 100
LR         = 1e-6                # smaller LR for the AUX formulation
EPOCHS     = 5_000               # bump to 250_000 for paper-quality results
N_LAYERS   = 5
FEATURES   = 128
R          = 128
OUT_DIM    = 2                   # (u, ν)
X_MIN, X_MAX = 0.0, 16 * np.pi
T_MAX      = 1.0


## SPINN model + HVP

In [ ]:
# Forward-over-forward HVP.  Used to compute u_xx, u_tt, u_xxxx, etc.
def hvp_fwdfwd(f, primals, tangents, return_primals=False):
    g = lambda primals: jvp(f, (primals,), tangents)[1]
    primals_out, tangents_out = jvp(g, primals, tangents)
    if return_primals:
        return primals_out, tangents_out
    return tangents_out


In [ ]:
# Separable PINN with 2 axes and `out_dim` separate output fields.
# Each output i uses its own block of size r in the last linear layer; we
# split the rank-(r * out_dim) tensor and merge with einsum.
class SPINN2dMulti(nn.Module):
    features: Sequence[int]
    r: int
    out_dim: int
    mlp: str = 'modified_mlp'

    @nn.compact
    def __call__(self, t, x):
        inputs, outputs, preds = [t, x], [], []
        init = nn.initializers.glorot_normal()
        for X in inputs:
            if self.mlp == 'mlp':
                for fs in self.features[:-1]:
                    X = nn.tanh(nn.Dense(fs, kernel_init=init)(X))
                X = nn.Dense(self.r * self.out_dim, kernel_init=init)(X)
            else:
                U = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                V = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                H = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                for fs in self.features[:-1]:
                    Z = nn.tanh(nn.Dense(fs, kernel_init=init)(H))
                    H = (1 - Z) * U + Z * V
                X = nn.Dense(self.r * self.out_dim, kernel_init=init)(H)
            outputs.append(jnp.transpose(X, (1, 0)))  # (r*out_dim, N_axis)
        for i in range(self.out_dim):
            a = outputs[0][self.r * i:self.r * (i + 1)]  # (r, T)
            b = outputs[1][self.r * i:self.r * (i + 1)]  # (r, Nx)
            preds.append(jnp.einsum('rt,rx->tx', a, b))
        return preds if self.out_dim > 1 else preds[0]


## Analytic solution & data generator (same as EB forward)

In [ ]:
def exact_u(t, x):
    return jnp.sin(x) * jnp.cos(jnp.pi * t)

def source_term(t, x):
    return 0.0

def make_train_data(nc):
    tc = jnp.linspace(0, T_MAX, nc + 2)[1:-1].reshape(-1, 1)
    xc = jnp.linspace(X_MIN, X_MAX, nc + 2)[1:-1].reshape(-1, 1)
    tm, xm = jnp.meshgrid(tc.ravel(), xc.ravel(), indexing='ij')
    uc = jnp.broadcast_to(source_term(tm, xm), tm.shape)

    ti = jnp.zeros((1, 1)); xi = xc
    ti_m, xi_m = jnp.meshgrid(ti.ravel(), xi.ravel(), indexing='ij')
    ui = exact_u(ti_m, xi_m)

    tb  = tc
    xbl = jnp.zeros((nc, 1))
    xbr = X_MAX * jnp.ones((nc, 1))
    tbl_m, xbl_m = jnp.meshgrid(tb.ravel(), xbl.ravel(), indexing='ij')
    tbr_m, xbr_m = jnp.meshgrid(tb.ravel(), xbr.ravel(), indexing='ij')
    ubl = exact_u(tbl_m, xbl_m)
    ubr = exact_u(tbr_m, xbr_m)

    W = jnp.tril(jnp.ones((nc, nc)), k=-1)
    return tc, xc, uc, ti, xi, ui, tb, xbl, xbr, ubl, ubr, W


## Causal loss with auxiliary consistency

In [ ]:
@partial(jax.jit, static_argnames=('apply_fn',))
def loss_and_grad(apply_fn, params, *train_data):
    tc, xc, uc, ti, xi, ui, tb, xbl, xbr, ubl, ubr, W = train_data

    def residual_loss(p):
        u, nu_ = apply_fn(p, tc, xc)
        v = jnp.ones(tc.shape)
        u_tt   = hvp_fwdfwd(lambda t: apply_fn(p, t, xc)[0], (tc,), (v,))
        u_xx   = hvp_fwdfwd(lambda x: apply_fn(p, tc, x)[0], (xc,), (v,))
        nu_xx  = hvp_fwdfwd(lambda x: apply_fn(p, tc, x)[1], (xc,), (v,))

        res1 = u_xx - nu_                        # consistency
        res2 = u_tt + nu_xx + (jnp.pi**2 - 1) * u - uc   # PDE

        lt1 = jnp.mean(res1**2, axis=1, keepdims=True).reshape(-1)
        lt2 = jnp.mean(res2**2, axis=1, keepdims=True).reshape(-1)
        agg1 = jax.lax.stop_gradient(jnp.dot(W, lt1))
        agg2 = jax.lax.stop_gradient(jnp.dot(W, lt2))
        cw1 = jnp.exp(-5.0 * agg1)
        cw2 = jnp.exp(-5.0 * agg2)
        return 10.0 * jnp.mean(cw1 * lt1) + jnp.mean(cw2 * lt2)

    def initial_loss(p):
        u0, _ = apply_fn(p, ti, xi)
        ic_disp = jnp.mean((u0 - ui)**2)
        v_t = jnp.ones(ti.shape)
        u_t0 = jvp(lambda t: apply_fn(p, t, xi)[0], (ti,), (v_t,))[1]
        ic_vel = jnp.mean(u_t0**2)
        return ic_disp + ic_vel

    def boundary_loss(p):
        v = jnp.ones(tb.shape)
        ul, _ = apply_fn(p, tb, xbl); ur, _ = apply_fn(p, tb, xbr)
        uxx_l = hvp_fwdfwd(lambda xbl: apply_fn(p, tb, xbl)[0], (xbl,), (v,))
        uxx_r = hvp_fwdfwd(lambda xbr: apply_fn(p, tb, xbr)[0], (xbr,), (v,))
        return (jnp.mean((ul - ubl)**2) + jnp.mean((ur - ubr)**2)
              + jnp.mean((uxx_l - ubr)**2) + jnp.mean((uxx_r - ubl)**2))

    total = lambda p: 0.1 * residual_loss(p) + initial_loss(p) + boundary_loss(p)
    return value_and_grad(total)(params)


## Initialize and train

In [ ]:
key = jax.random.PRNGKey(SEED)
key, sub_init = jax.random.split(key, 2)

model = SPINN2dMulti(features=[FEATURES] * N_LAYERS, r=R, out_dim=OUT_DIM)
t0 = jnp.ones((NC, 1)); x0 = jnp.ones((NC, 1))
params = model.init(sub_init, t0, x0)
apply_fn = jax.jit(model.apply)

n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total trainable params: {n_params}")

train_data = make_train_data(NC)

# Eval grid + ground truth for periodic best-tracking
t_eval = jnp.linspace(0, T_MAX, 100).reshape(-1, 1)
x_eval = jnp.linspace(X_MIN, X_MAX, 100).reshape(-1, 1)
tm_eval, xm_eval = jnp.meshgrid(t_eval.ravel(), x_eval.ravel(), indexing='ij')
u_true_eval = exact_u(tm_eval, xm_eval)

optim = optax.adam(LR)
state = optim.init(params)

losses = []
best_loss = 1e9
best_err = 1e9
best_params = params

pbar = trange(EPOCHS)
for e in pbar:
    loss, grads = loss_and_grad(apply_fn, params, *train_data)
    updates, state = optim.update(grads, state, params)
    params = optax.apply_updates(params, updates)
    losses.append(float(loss))
    if float(loss) <= best_loss:                       
        best_loss = float(loss)
        u_pred_eval, _ = apply_fn(params, t_eval, x_eval)
        best_err = float(jnp.linalg.norm(u_pred_eval - u_true_eval) / jnp.linalg.norm(u_true_eval))
        best_params = params
    if (e + 1) % 500 == 0:
        pbar.set_postfix(loss=f"{loss:.3e}", err_at_best_loss=f"{best_err:.3e}")

print(f"\nRel L2 error (u) at best-loss step: {best_err:.3e}")


## Evaluate and plot (using BEST params)

In [ ]:
# Relative L2 error against the analytic solution.
def relative_l2(pred, true):
    return float(jnp.linalg.norm(pred - true) / jnp.linalg.norm(true))

# AUX model returns (u, ν); use only u for visualization
t_test = jnp.linspace(0, T_MAX, 100).reshape(-1, 1)
x_test = jnp.linspace(X_MIN, X_MAX, 100).reshape(-1, 1)
tm, xm = jnp.meshgrid(t_test.ravel(), x_test.ravel(), indexing='ij')
u_true = exact_u(tm, xm)
u_pred, nu_pred = apply_fn(best_params, t_test, x_test)
err = best_err
print(f"Best relative L2 error (u): {err:.3e}")

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title in zip(
        axs,
        [u_true, u_pred, jnp.abs(u_true - u_pred)],
        ['Exact $u$', 'Predicted $\hat u$', '|Error|']):
    im = ax.pcolormesh(np.asarray(tm), np.asarray(xm), np.asarray(data),
                       cmap='RdBu_r', shading='auto')
    ax.set_xlabel('t'); ax.set_ylabel('x'); ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.semilogy(losses); plt.xlabel('epoch'); plt.ylabel('total loss')
plt.title('EB AUX Causal SPINN — training loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
